# 2.1) Implementing a custom apodization function

This example demonstrates how to implement a custom apodization function leveraging
`pyscopee`.

For this purpose, the [Cosine window](https://en.wikipedia.org/wiki/Window_function#Sine_window)
and the [Kaiser window](https://en.wikipedia.org/wiki/Window_function#Kaiser_window)
will be implemented.

Let's first start out by import `matplotlib.pyplot`, `numpy`, and `pyscopee`.

## Imports


In [1]:
import numpy as np
import pyscopee as psc
from matplotlib import pyplot as plt

psc.apply_pyscopee_plot_style()

%matplotlib widget

## Cosine window

Over the interval $\left[-1, 1\right]$, the Cosine window is defined as:

$f\left(x\right) = \cos\left(\frac{\pi\cdot x}{2}\right)$

Let's start by implementing this function and visualising it together with the
boundaries at $x=-1$ and $x=1$.


In [ ]:
def cosine_simple(x: np.ndarray) -> np.ndarray:
    return np.cos((0.5 * np.pi) * x)


x = np.linspace(
    start=-3.5,
    stop=3.5,
    num=1_001,
)

try:
    plt.close(fig)  # type: ignore
except NameError:
    pass

fig, ax = plt.subplots(
    figsize=(12, 6),
)

ax.axhline(0.0, color="black")
ax.axvline(0.0, color="black")

label = "Boundary"
for boundary in (-1.0, 1.0):
    ax.axvline(boundary, color="black", linestyle="--", label=label)
    label = None  # to avoid 2 labels for the boundaries

ax.plot(
    x,
    cosine_simple(x),
    label="Cosine Apodization",
)

ax.set_xlabel("x")
ax.set_ylabel(r"$f\left(x\right)$")
ax.legend()

ax.set_xlim(x[0], x[-1])

pass

Well, that is already quite okay, but the apodization function is not zero outside
the boundaries. On top of that, the boundaries are currently $x=-1$ and $x=1$,
but what if they should be scaled to $x=-0.5$ and $x=0.5$?

Theoretically, this could be fixed by normalizing the $x$ values to the interval
$\left[-1, 1\right]$ and multiplying the function by a rectangular function
that is zero outside of $\left[-1, 1\right]$.

However, this is not necessary, as `pyscopee` provides the `as_apodization_function`
decorator that takes care of this for us. With this, we can simply define the
function over the interval $\left[0, 1\right]$ and `pyscopee` will take care of
the rest.

Admittedly, implementing a function for this decorator from scratch is not
trivial because the interface is highly standardised. Thus, `pyscopee` provides
the function `print_apodization_function_template` that prints a template from which
the function can be implemented.

Parts where **custom code** needs to be added are highlighted in **bold** while 
<ins>important remarks</ins> are <ins>underlined</ins>.

**Note:** In Jupyter, the print output probably needs to be expanded to see the
full template.

In [ ]:
psc.print_apodization_function_template(
    name="cosine",
    with_imports=True,
)

Alright, let's copy and paste the template and implement the Cosine window as an
apodization function.

It can immediately be decorated with the `@as_apodization_function` decorator
which will validate the function for correctness.

In [4]:
# === Imports ===

import numpy as np
from numpy.typing import NDArray

import pyscopee as psc

# === Function ===

@psc.as_apodization_function
def cosine(
    x: psc.RealNumericArrayLike,
    x_max: psc.RealNumeric = 1.0,
    *,
    # Insert additional keyword-only arguments here.
    skip_validation: bool = False,
) -> NDArray[np.float64]:
    """
    Computes the values of the cosine apodization function at the given points.

    Parameters
    ----------
    x : Array-like of shape (n,)
        The points at which to evaluate the apodization function.
        Negative entries are converted to positive ones under the assumption that
        the apodization function has even symmetry.
        Its length has to be at least 1.
        It is internally promoted to ``np.float64``.
    x_max : :class:`float` or :class:`int`, default=``1.0``
        The maximum value of the x-range over which the apodization function is
        defined.
        It must be a positive real number ``> 0``.
        With this, the x-range of ``[-1, 1]`` where apodization functions are
        typically defined is scaled to ``[-x_max, x_max]``.
    skip_validation : :class:`bool`, default=``False`` (keyword-only)
        Whether to skip the input validation of ``x_max`` (``True``) or not
        (``False``).
        ``x`` is always validated.
        This variable is meant for internal use only and it is highly
        discouraged to set it to ``True``.

    Returns
    -------
    apodization_values : :class:`numpy.ndarray` of shape (n,) of dtype ``np.float64``
        The values of the apodization function at the given points.

    Notes
    -----
    The cosine apodization function is defined as

    ```
    f(x) = cos((π / 2) * (x / x_max))
    ```

    within the interval ``[-x_max, x_max]``.

    """  # noqa: E501

    # --- Input Validation ---

    if not skip_validation:
        pass

    # --- Computation ---

    return np.cos((0.5 * np.pi) * x)  # type: ignore

Now, the apodization function can be visualised again.

In [ ]:
try:
    plt.close(fig2)  # type: ignore
except NameError:
    pass

fig2, ax2 = plt.subplots(
    figsize=(12, 6),
)

ax2.axhline(0.0, color="black")
ax2.axvline(0.0, color="black")

ax2.plot(
    x,
    cosine(x, x_max=1.0),
    label="(x_max=1.0)",
)
ax2.plot(
    x,
    cosine(x, x_max=0.5),
    label="(x_max=0.5)",
)

ax2.set_xlabel("x")
ax2.set_ylabel(r"$f\left(x\right)$")
ax2.legend()

ax2.set_xlim(x[0], x[-1])

fig2.suptitle("Cosine Apodization Function")

pass

As we can see, our apodization function is now zero outside of the interval
and fully flexible in terms of the boundaries 🎉🥳

Using the initial simple implementation of the Cosine window would not work and 
`pyscopee` would raise an error pointing to the issue.

**Note:** In Jupyter, the console output probably needs to be expanded to see the
full error message.

In [ ]:
psc.as_apodization_function(cosine_simple)

## Kaiser window

The Kaiser window is defined as:

$f\left(x\right) = \frac{I_0\left(\beta\cdot\sqrt{1-x^{2}}\right)}{I_0\left(\beta\right)}$

where $I_0$ is the modified Bessel function of the first kind of order zero and
$\beta$ is a parameter that controls the shape of the window.

In contrast to the Cosine window, the Kaiser window requires an additional parameter
$\beta$ that needs to be passed to the function. This is not as complicated as it
sounds as long as `beta` is defined as a keyword-only argument.

Again, we copy and paste the template and implement the Kaiser window as an
apodization function.

In [ ]:
psc.print_apodization_function_template(
    name="kaiser",
    with_imports=False,
)

In [9]:
# === Imports ===

from scipy.special import i0

# === Function ===

@psc.as_apodization_function
def kaiser(
    x: psc.RealNumericArrayLike,
    x_max: psc.RealNumeric = 1.0,
    *,
    beta: psc.RealNumeric = 14.0,
    skip_validation: bool = False,
) -> NDArray[np.float64]:
    """
    Insert the description of the apodization function here.

    Parameters
    ----------
    x : Array-like of shape (n,)
        The points at which to evaluate the apodization function.
        Negative entries are converted to positive ones under the assumption that
        the apodization function has even symmetry.
        Its length has to be at least 1.
        It is internally promoted to ``np.float64``.
    x_max : :class:`float` or :class:`int`, default=``1.0``
        The maximum value of the x-range over which the apodization function is
        defined.
        It must be a positive real number ``> 0``.
        With this, the x-range of ``[-1, 1]`` where apodization functions are
        typically defined is scaled to ``[-x_max, x_max]``.
    beta : :class:`float` or :class:`int`, default=``14.0`` (keyword-only)
        The shape parameter of the Kaiser window.
        It must be a non-negative real number ``>= 0``.
        Please refer to the Notes section for typical values and their corresponding
        related apodization functions.
    skip_validation : :class:`bool`, default=``False`` (keyword-only)
        Whether to skip the input validation of ``x_max`` (``True``) or not
        (``False``).
        ``x`` is always validated.
        This variable is meant for internal use only and it is highly
        discouraged to set it to ``True``.

    Returns
    -------
    apodization_values : :class:`numpy.ndarray` of shape (n,) of dtype ``np.float64``
        The values of the apodization function at the given points.

    Notes
    -----
    The Kaiser apodization function is defined as

    ```
    f(x) = I_0(beta * sqrt(1 - (x / x_max) ** 2)) / I_0(beta)
    ```

    within the interval ``[-x_max, x_max]``.

    Typical values of ``beta`` and their corresponding related apodization functions are

    - ``beta = 0``: Rectangular or boxcar apodization
    - ``beta = 5``: similar to the Hamming window
    - ``beta = 6``: similar to the Hann window
    - ``beta = 8.6``: similar to the Blackman window


    """  # noqa: E501

    # --- Input Validation ---

    if not skip_validation:
        beta = psc.get_validated_real_numeric(
            value=beta,
            name="beta",
            min_value=0.0,
            min_inclusive=True
        )

    # --- Computation ---

    return (1.0 / i0(beta)) * i0(beta * np.sqrt(1.0 - np.square(x)))  # type: ignore


Again, the apodization function can be visualised.

In [ ]:
try:
    plt.close(fig3)  # type: ignore
except NameError:
    pass

fig3, ax3 = plt.subplots(
    figsize=(12, 6),
)

ax3.axhline(0.0, color="black")
ax3.axvline(0.0, color="black")

ax3.plot(
    x,
    kaiser(x, x_max=0.5, beta=0.0),
    label="(x_max=0.5, beta=0.0)",
)
ax3.plot(
    x,
    kaiser(x, x_max=0.75, beta=5.0),
    label="(x_max=0.75, beta=5.0)",
)
ax3.plot(
    x,
    kaiser(x, x_max=1.5, beta=6.0),
    label="(x_max=1.5, beta=6.0)",
)
ax3.plot(
    x,
    kaiser(x, x_max=2.0, beta=8.6),
    label="(x_max=2.0, beta=8.6)",
)
ax3.plot(
    x,
    kaiser(x, x_max=2.5, beta=14.0),
    label="(x_max=2.5, beta=14.0)",
)

ax3.set_xlabel("x")
ax3.set_ylabel(r"$f\left(x\right)$")
ax3.legend()

ax3.set_xlim(x[0], x[-1])

fig3.suptitle("Kaiser Apodization Function")

pass

Now, the Kaiser window is also implemented as an apodization function and fully flexible
in terms of the `x_max` and `beta` parameters 🎉🥳

In case some input to the function is invalid, `pyscopee` will raise an error that will
help to identify the issue.

**Note:** In Jupyter, the console output probably needs to be expanded to see the
full error message.

In [ ]:
kaiser(
    x=np.array([0.0, 0.5, 1.0]),
    x_max=-10.0,  # this will cause an error
    beta=5.0,
)